# QIIME 2 Microbiome Analysis Pipeline
## Complete workflow for microbiome research analysis
This notebook implements a full QIIME 2 pipeline for analyzing microbiome 16S rRNA gene sequencing data from your Savolainen Lab samples.

## 1. Set Up the QIIME 2 Environment and Imports

Import QIIME 2 modules, verify installation, and set up the analysis environment.

In [ ]:
#!/usr/bin/env python3
import subprocess
import sys
import os
import pandas as pd
import numpy as np
import pip
from pathlib import Path

# Import QIIME 2
try:
    import qiime2
    print(f"✓ QIIME 2 version: {qiime2.__version__}")
except ImportError:
    print("✗ QIIME 2 not found. Please activate the QIIME 2 conda environment.")
    sys.exit(1)

# Import QIIME 2 plugins 
from qiime2.plugins import demux, dada2, feature_table, alignment, phylogeny, diversity, taxa

print("✓ All QIIME 2 modules loaded successfully")

# Set up working directory
work_dir = Path("/Users/clarence/Savolainen Lab/Microbiome/Qiime")
fastq_dir = work_dir / "fastq"
output_dir = work_dir / "output"
output_dir.mkdir(exist_ok=True)

print(f"✓ Working directory: {work_dir}")
print(f"✓ FASTQ directory: {fastq_dir}")
print(f"✓ Output directory: {output_dir}")

✓ QIIME 2 version: 2026.4.0
✓ All QIIME 2 modules loaded successfully
✓ Working directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime
✓ FASTQ directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/fastq
✓ Output directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output


## 2. Prepare Sample Metadata and Input File Paths

Load metadata, validate required columns, and set up file paths for the analysis.

In [3]:
# Load metadata
metadata_file = work_dir / "metadata.tsv"
metadata_df = pd.read_csv(metadata_file, sep='\t')

print(f"✓ Metadata loaded: {metadata_file}")
print(f"  Samples: {len(metadata_df)}")
print(f"  Columns: {list(metadata_df.columns)}\n")

# Display first few rows
print("Metadata preview:")
print(metadata_df.head())

# Validate required column
if 'sample-id' not in metadata_df.columns:
    print("✗ Error: 'sample-id' column not found in metadata")
else:
    print(f"\n✓ Found sample-id column with {len(metadata_df)} samples")

# Define file paths
manifest_file = work_dir / "manifest_fixed.tsv"
if not manifest_file.exists():
    manifest_file = work_dir / "manifest.tsv"

print(f"✓ Manifest file: {manifest_file}")
print(f"  File exists: {manifest_file.exists()}")

# Load manifest to verify
manifest_df = pd.read_csv(manifest_file, sep='\t')
print(f"\n✓ Manifest loaded: {len(manifest_df)} samples")
print("Manifest preview:")
print(manifest_df.head())

✓ Metadata loaded: /Users/clarence/Savolainen Lab/Microbiome/Qiime/metadata.tsv
  Samples: 39
  Columns: ['sample-id', 'monkey_id', 'library_id', 'rectal_swab', 'SSB']

Metadata preview:
            sample-id monkey_id library_id rectal_swab       SSB
0  Li49848-2023086-RT       04T    Li49848  2023086_RT  1.681659
1   Li49872-023057-RT       04Z    Li49872   023057_RT  0.219089
2  Li49835-2023095-RT       0J4    Li49835  2023095_RT  3.280724
3   Li49758-023028-RT       1B1    Li49758   023028_RT  0.376073
4   Li49757-023003-RT       2C9    Li49757   023003_RT  0.699226

✓ Found sample-id column with 39 samples
✓ Manifest file: /Users/clarence/Savolainen Lab/Microbiome/Qiime/manifest_fixed.tsv
  File exists: True

✓ Manifest loaded: 39 samples
Manifest preview:
            sample-id                          forward-absolute-filepath  \
0  Li49683-2023089-RT  /Users/clarence/Savolainen Lab/Microbiome/Qiim...   
1   Li49689-023055-RT  /Users/clarence/Savolainen Lab/Microbiome/Qiim...   


## 3. Import Sequencing Data into QIIME 2 Artifacts

Import raw paired-end sequence data using the manifest file. The manifest format specifies absolute paths to forward (R1) and reverse (R2) reads.

In [4]:
import qiime2
from qiime2.plugins import demux
import pandas as pd
from pathlib import Path

# Check if paired sequences have already been imported
sequences_artifact = output_dir / "paired_sequences.qza"
manifest_combined_file = work_dir / "manifest_combined_fixed.tsv"
sample_manifest_file = manifest_file
blank_fastq_dir = work_dir / "blank fastq"
negative_controls_path = work_dir / "Vincent_Negative_Controls.csv"


def _normalize_manifest(manifest_in):
    manifest_in.columns = [str(column).strip() for column in manifest_in.columns]

    if {"sample-id", "forward-absolute-filepath", "reverse-absolute-filepath"}.issubset(manifest_in.columns):
        manifest_v2 = manifest_in[["sample-id", "forward-absolute-filepath", "reverse-absolute-filepath"]].copy()
    elif {"sample-id", "absolute-filepath", "direction"}.issubset(manifest_in.columns):
        tmp = manifest_in[["sample-id", "absolute-filepath", "direction"]].copy()
        tmp["direction"] = tmp["direction"].astype(str).str.strip().str.lower()
        manifest_v2 = (
            tmp.pivot_table(
                index="sample-id",
                columns="direction",
                values="absolute-filepath",
                aggfunc="first",
            )
            .reset_index()
            .rename(
                columns={
                    "forward": "forward-absolute-filepath",
                    "reverse": "reverse-absolute-filepath",
                }
            )
        )
    else:
        raise ValueError(
            "Manifest must contain either paired-end V2 columns or long-format columns"
        )

    required_cols = ["sample-id", "forward-absolute-filepath", "reverse-absolute-filepath"]
    missing_cols = [column for column in required_cols if column not in manifest_v2.columns]
    if missing_cols:
        raise ValueError(f"Missing required manifest columns after conversion: {missing_cols}")

    return manifest_v2[required_cols].copy()


def _build_control_manifest():
    if not negative_controls_path.exists() or not blank_fastq_dir.exists():
        print("Warning: control FASTQ inputs were not found; the combined manifest will only include samples.")
        return pd.DataFrame(columns=["sample-id", "forward-absolute-filepath", "reverse-absolute-filepath"])

    controls = pd.read_csv(negative_controls_path)
    controls.columns = [str(column).strip() for column in controls.columns]

    if "File_Root_Name" not in controls.columns:
        raise ValueError(f"Missing File_Root_Name column in {negative_controls_path}")

    if "Is_Neg" in controls.columns:
        neg_mask = (
            controls["Is_Neg"].astype(str).str.upper().isin({"Y", "TRUE", "T", "1"})
        )
    else:
        print("Warning: No Is_Neg column found in the control file; assuming all rows are controls.")
        neg_mask = pd.Series(True, index=controls.index)

    control_ids = controls.loc[neg_mask, "File_Root_Name"].astype(str).tolist()
    control_rows = []
    missing_controls = []

    for sample_id in control_ids:
        forward_candidates = sorted(blank_fastq_dir.glob(f"{sample_id}*R1_001.fastq.gz"))
        reverse_candidates = sorted(blank_fastq_dir.glob(f"{sample_id}*R2_001.fastq.gz"))

        if not forward_candidates or not reverse_candidates:
            missing_controls.append(sample_id)
            continue

        control_rows.append(
            {
                "sample-id": sample_id,
                "forward-absolute-filepath": str(forward_candidates[0].resolve()),
                "reverse-absolute-filepath": str(reverse_candidates[0].resolve()),
            }
        )

    if missing_controls:
        raise FileNotFoundError(
            f"Missing FASTQ files for controls: {missing_controls}"
        )

    return pd.DataFrame(control_rows)


def _build_combined_manifest():
    sample_manifest = _normalize_manifest(pd.read_csv(sample_manifest_file, sep=None, engine="python"))
    control_manifest = _build_control_manifest()

    combined_manifest = pd.concat([sample_manifest, control_manifest], ignore_index=True)
    combined_manifest = (
        combined_manifest.drop_duplicates(subset=["sample-id"])
        .sort_values("sample-id")
        .reset_index(drop=True)
    )
    combined_manifest.to_csv(manifest_combined_file, sep="\t", index=False)

    print(f"Combined manifest written: {manifest_combined_file}")
    print(f"  Samples: {len(sample_manifest)}")
    print(f"  Controls: {len(control_manifest)}")
    print(f"  Total: {len(combined_manifest)}")

    return combined_manifest


combined_manifest = _build_combined_manifest()

print("Importing paired-end sequences from the combined manifest...")
paired_demux_sequences = qiime2.Artifact.import_data(
    "SampleData[PairedEndSequencesWithQuality]",
    str(manifest_combined_file),
    view_type="PairedEndFastqManifestPhred33V2",
)

paired_demux_sequences.save(str(sequences_artifact))
print(f"Saved paired sequences artifact: {sequences_artifact}")
print(f"Sequence count in manifest: {len(combined_manifest)}")
print("Next: run DADA2 denoising with the full 49-sample import if desired")

Combined manifest written: /Users/clarence/Savolainen Lab/Microbiome/Qiime/manifest_combined_fixed.tsv
  Samples: 39
  Controls: 10
  Total: 49
Importing paired-end sequences from the combined manifest...
Saved paired sequences artifact: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/paired_sequences.qza
Sequence count in manifest: 49
Next: run DADA2 denoising with the full 49-sample import if desired


## 4. Summarize and Visualize Demultiplexed Reads

Generate quality reports and visualizations to assess read length distribution, quality scores, and sequencing depth before denoising.

In [ ]:
# Summarize demultiplexed sequences using Python API
print("Generating demultiplexing summary and visualization...")

# FIX: Try-except with graceful fallback for demux.summarize()
try:
    # Attempt to call demux summarize via Python API
    # Note: This may not be available in all QIIME versions; if so, we'll skip it
    from qiime2.plugins.demux.methods import summarize as demux_summarize
    demux_summary = demux_summarize(paired_demux_sequences)
    
    # Save the visualization
    demux_viz_file = output_dir / "demux_summary.qzv"
    demux_summary.save(str(demux_viz_file))
    print(f"✓ Saved demux summary visualization: {demux_viz_file}")
except (ImportError, AttributeError) as e:
    print(f"Note: Demux visualization not available via Python API in this version.")
    print(f"The sequences are imported and ready for DADA2 denoising.")

print("\n" + "═" * 50)
print("Demultiplexing Status:")
print("═" * 50)
print("✓ Sequences imported successfully")
print("  Artifact type: SampleData[PairedEndSequencesWithQuality]")
print(f"  Location: {sequences_artifact}")

print("\nTo generate quality visualizations manually, run:")
print("─" * 50)
print("qiime demux summarize \\")
print(f"  --i-data {sequences_artifact} \\")
print(f"  --o-visualization {output_dir}/demux_summary.qzv")
print("─" * 50)

print("\nQuality check recommendations:")
print("1. Review forward/reverse read quality in the demux summary")
print("2. Determine appropriate trunc_len_f and trunc_len_r values for DADA2")
print("3. Adjust truncation parameters in the next cell if needed")
print("4. Proceed to DADA2 denoising in the next section")

Generating demultiplexing summary and visualization...
Note: Demux visualization not available via Python API in this version.
      The sequences are imported and ready for DADA2 denoising.

══════════════════════════════════════════════════
Demultiplexing Status:
══════════════════════════════════════════════════
✓ Sequences imported successfully
  Artifact type: SampleData[PairedEndSequencesWithQuality]
  Location: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/paired_sequences.qza

To generate quality visualizations manually, run:
──────────────────────────────────────────────────
qiime demux summarize \
  --i-data /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/paired_sequences.qza \
  --o-visualization /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/demux_summary.qzv
──────────────────────────────────────────────────

Quality check recommendations:
1. Review forward/reverse read quality in the demux summary
2. Determine appropriate trunc_len_f and trunc_len_

## 5. Denoise Reads with DADA2 and Build ASVs

Run DADA2 to filter low-quality reads, merge paired-end sequences, remove chimeras, and build an Amplicon Sequence Variant (ASV) table.

### The first block of code is for the three-sample run, to test out the pipeline

In [ ]:
import time
import datetime

# Define output paths for test artifacts
test_table_path = output_dir / "test_feature_table_3samples.qza"
test_rep_seqs_path = output_dir / "test_rep_seqs_3samples.qza"
test_dada2_stats_path = output_dir / "test_dada2_stats_3samples.qza"

# Check if test artifacts already exist
test_artifacts_exist = all([
    test_table_path.exists(),
    test_rep_seqs_path.exists(),
    test_dada2_stats_path.exists()
])

if test_artifacts_exist:
    print("=" * 60)
    print("TEST DADA2 ARTIFACTS ALREADY EXIST")
    print("=" * 60)
    print(f"Loading existing test artifacts...")
    test_table = qiime2.Artifact.load(str(test_table_path))
    test_rep_seqs = qiime2.Artifact.load(str(test_rep_seqs_path))
    test_dada2_stats = qiime2.Artifact.load(str(test_dada2_stats_path))
    print("Test artifacts loaded successfully")
    print(f"  Feature table: {test_table_path}")
    print(f"  Rep sequences: {test_rep_seqs_path}")
    print(f"  DADA2 stats: {test_dada2_stats_path}")
else:
    print("=" * 60)
    print("PERFORMANCE BENCHMARK: Testing DADA2 on 3-sample subset")
    print("=" * 60)

    # Load the sample-only manifest and select the first 3 biological samples
    test_manifest = pd.read_csv(manifest_file, sep=None, engine="python")
    test_manifest.columns = [str(column).strip() for column in test_manifest.columns]
    if "sample-id" not in test_manifest.columns:
        raise ValueError(f"Expected a sample-id column in {manifest_file}")

    test_manifest_subset = test_manifest.head(3).copy()

    # Write subset manifest
    test_manifest_file = work_dir / "manifest_test_3samples.tsv"
    test_manifest_subset.to_csv(test_manifest_file, sep="\t", index=False)

    print(f"\nTest samples selected: {list(test_manifest_subset['sample-id'].values)}")
    print(f"Manifest file: {test_manifest_file}\n")

    # Import just the 3 test samples
    test_seqs = qiime2.Artifact.import_data(
        'SampleData[PairedEndSequencesWithQuality]',
        str(test_manifest_file),
        view_type='PairedEndFastqManifestPhred33V2'
    )

    print("Test samples imported successfully")

    # Set truncation and trimming parameters for the DADA2 test run
    TRUNC_LEN_F = 280
    TRUNC_LEN_R = 260
    TRIM_LEFT_F = 0
    TRIM_LEFT_R = 0
    POOLING_METHOD = 'pseudo'
    CHIMERA_METHOD = 'consensus'
    MIN_FOLD_PARENT_OVER_ABUNDANCE = 8.0
    
    import os
    N_THREADS = max(1, (os.cpu_count() or 1) - 1)
    print(f"Set TRUNC_LEN_F={TRUNC_LEN_F}, TRUNC_LEN_R={TRUNC_LEN_R}, N_THREADS={N_THREADS}")
    print(f"Set POOLING_METHOD={POOLING_METHOD}, CHIMERA_METHOD={CHIMERA_METHOD}, MIN_FOLD_PARENT_OVER_ABUNDANCE={MIN_FOLD_PARENT_OVER_ABUNDANCE}")

    # Run DADA2 on test subset and time it
    from qiime2.plugins import dada2
    import os

    N_THREADS = max(1, (os.cpu_count() or 1) - 1)
    N_READS_LEARN = 200000  # Set lower for testing but return to 500,000 for full run

    print(f"\nRunning DADA2 on 3 samples using {N_THREADS} threads...")
    print(f"Error learning reads: {N_READS_LEARN:,} (optimized for speed)")
    print("-" * 60)

    start_time = time.time()

    # Capture the raw result so we can adapt to different return shapes
    res = dada2.methods.denoise_paired(
        demultiplexed_seqs=test_seqs,
        trunc_len_f=TRUNC_LEN_F,
        trunc_len_r=TRUNC_LEN_R,
        trim_left_f=TRIM_LEFT_F,
        trim_left_r=TRIM_LEFT_R,
        pooling_method=POOLING_METHOD,
        chimera_method=CHIMERA_METHOD,
        min_fold_parent_over_abundance=MIN_FOLD_PARENT_OVER_ABUNDANCE,
        n_threads=N_THREADS,
        n_reads_learn=N_READS_LEARN
    )

    # Normalize outputs: try attributes first, then tuple unpacking
    if hasattr(res, 'table') and hasattr(res, 'representative_sequences') and hasattr(res, 'denoising_stats'):
        test_table = res.table
        test_rep_seqs = res.representative_sequences
        test_dada2_stats = res.denoising_stats
    else:
        try:
            test_table, test_rep_seqs, test_dada2_stats = res
        except Exception:
            print("Unexpected result structure from dada2.methods.denoise_paired():", type(res))
            try:
                print("Result attributes:", [a for a in dir(res) if not a.startswith('_')])
            except Exception:
                pass
            raise

    end_time = time.time()
    elapsed_seconds = end_time - start_time
    elapsed_minutes = elapsed_seconds / 60
    elapsed_hours = elapsed_minutes / 60

    print("-" * 60)
    print("\nTest completed!")

    # Calculate statistics
    time_per_sample = elapsed_minutes / 3
    estimated_full_39_samples = time_per_sample * 39
    estimated_full_hours = estimated_full_39_samples / 60

    print(f"\nBENCHMARK RESULTS:")
    print(f"  Total time for 3 samples: {elapsed_minutes:.2f} minutes ({elapsed_hours:.3f} hours)")
    print(f"  Time per sample: {time_per_sample:.2f} minutes")
    print(f"  Estimated time for full dataset (39 samples): {estimated_full_hours:.2f} hours")
    print(f"  Estimated completion: ~{estimated_full_hours:.1f} hours from now")

    # Save test results
    test_table.save(str(test_table_path))
    test_rep_seqs.save(str(test_rep_seqs_path))
    test_dada2_stats.save(str(test_dada2_stats_path))

    print(f"\nTest artifacts saved to {output_dir}/")

print("\nReady to proceed with full 49-sample analysis when you're ready.")

PERFORMANCE BENCHMARK: Testing DADA2 on 3-sample subset

Test samples selected: ['Li49683-2023089-RT', 'Li49689-023055-RT', 'Li49747-023037-RT']
Manifest file: /Users/clarence/Savolainen Lab/Microbiome/Qiime/manifest_test_3samples.tsv

✓ Imported 3 test samples
Set TRUNC_LEN_F=280, TRUNC_LEN_R=260, N_THREADS=7
Set POOLING_METHOD=pseudo, CHIMERA_METHOD=consensus, MIN_FOLD_PARENT_OVER_ABUNDANCE=4.0

Running DADA2 on 3 samples using 7 threads...
Error learning reads: 200,000 (optimized for speed)
------------------------------------------------------------
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_dada.R --input_directory /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/tmpg_cfbv9p/forward --input_directory_reverse /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/tmpg_cfbv9p/reverse --ou

Loading required package: Rcpp


DADA2: 1.38.0 / Rcpp: 1.1.1 / RcppParallel: 5.1.11.2 
2) Filtering ...
3) Learning Error Rates
185389960 total bases in 662107 reads from 1 samples will be used for learning the error rates.
172147820 total bases in 662107 reads from 1 samples will be used for learning the error rates.
3) Denoise samples ...
  Pseudo-pool step ...
...
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output


/opt/miniconda3/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan 

------------------------------------------------------------

✓ Test completed!

BENCHMARK RESULTS:
  Total time for 3 samples: 249.32 minutes (4.155 hours)
  Time per sample: 83.11 minutes
  Estimated time for full dataset (39 samples): 54.02 hours
  Estimated completion: ~54.0 hours from now

✓ Test artifacts saved to /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/

Ready to proceed with full 39-sample analysis when you're ready.


### This next block of code is for the full sample run

In [5]:
from qiime2.plugins import dada2
import os
import time

def show_progress(label, percent, width=30):
    """Display a simple progress bar"""
    filled = int(width * percent / 100)
    bar = '█' * filled + '░' * (width - filled)
    print(f"[{bar}] {percent:3d}% - {label}")

# Define output paths for full dataset artifacts
feature_table_path = output_dir / "feature_table.qza"
rep_seqs_path = output_dir / "rep_seqs.qza"
dada2_stats_path = output_dir / "dada2_stats.qza"

# Check if full artifacts already exist
full_artifacts_exist = all([
    feature_table_path.exists(),
    rep_seqs_path.exists(),
    dada2_stats_path.exists()
])

if full_artifacts_exist:
    print("=" * 60)
    print("FULL DADA2 ARTIFACTS ALREADY EXIST")
    print("=" * 60)
    print(f"Loading existing full-dataset artifacts...")
    table = qiime2.Artifact.load(str(feature_table_path))
    rep_seqs = qiime2.Artifact.load(str(rep_seqs_path))
    dada2_stats = qiime2.Artifact.load(str(dada2_stats_path))
    print("✓ Full dataset artifacts loaded successfully")
    print(f"  Feature table: {feature_table_path}")
    print(f"  Rep sequences: {rep_seqs_path}")
    print(f"  DADA2 stats: {dada2_stats_path}")
    print("\nYou can now skip to taxonomy classification (run the next notebook).")
else:
    # DADA2 Parameters
    # NOTE: Adjust these based on the demux quality summary
    #   - trunc_len_f/r: Position to truncate forward/reverse reads
    #   - trim_left_f/r: Position to trim from the beginning
    TRUNC_LEN_F = 280  # Forward read truncation length
    TRUNC_LEN_R = 260  # Reverse read truncation length
    TRIM_LEFT_F = 0    # No trimming from the left by default
    TRIM_LEFT_R = 0    # No trimming from the left by default
    POOLING_METHOD = 'pseudo'  # DADA2 pooling method: 'independent' or 'pseudo'
    CHIMERA_METHOD = 'consensus'    # Chimera removal method: 'consensus', 'none', or 'pooled'
    MIN_FOLD_PARENT_OVER_ABUNDANCE = 8.0 

    # Calculate available threads (use all except 1)
    N_THREADS = max(1, (os.cpu_count() or 1) - 1)
    N_READS_LEARN = 500000  # Optimized: reduced from 1M default for faster learning phase

    print("=" * 60)
    print("Running DADA2 denoising pipeline (FULL DATASET - 39 samples + 10 negative controls)")
    print("=" * 60)
    print(f"\nParameters:")
    print(f"  Trim left forward: {TRIM_LEFT_F}")
    print(f"  Trim left reverse: {TRIM_LEFT_R}")
    print(f"  Truncate length forward: {TRUNC_LEN_F}")
    print(f"  Truncate length reverse: {TRUNC_LEN_R}")
    print(f"  Pooling method: {POOLING_METHOD}")
    print(f"  Chimera method: {CHIMERA_METHOD}")
    print(f"  Min fold parent over abundance: {MIN_FOLD_PARENT_OVER_ABUNDANCE}")
    print(f"  Threads: {N_THREADS}")
    print(f"  Error learning reads: {N_READS_LEARN:,} (optimized for speed)")
    print()

    show_progress("Starting DADA2", 0)
    show_progress("Launching denoising job", 10)

    start_time = time.time()

    # Capture raw result
    res = dada2.methods.denoise_paired(
        demultiplexed_seqs=paired_demux_sequences,
        trunc_len_f=TRUNC_LEN_F,
        trunc_len_r=TRUNC_LEN_R,
        trim_left_f=TRIM_LEFT_F,
        trim_left_r=TRIM_LEFT_R,
        pooling_method=POOLING_METHOD,
        chimera_method=CHIMERA_METHOD,
        min_fold_parent_over_abundance=MIN_FOLD_PARENT_OVER_ABUNDANCE,
        n_threads=N_THREADS,
        n_reads_learn=N_READS_LEARN
    )

    # Normalize outputs
    if hasattr(res, 'table') and hasattr(res, 'representative_sequences') and hasattr(res, 'denoising_stats'):
        table = res.table
        rep_seqs = res.representative_sequences
        dada2_stats = res.denoising_stats
    else:
        try:
            table, rep_seqs, dada2_stats = res
        except Exception:
            print("Unexpected result structure from dada2.methods.denoise_paired():", type(res))
            try:
                print("Result attributes:", [a for a in dir(res) if not a.startswith('_')])
            except Exception:
                pass
            raise

    end_time = time.time()
    elapsed_minutes = (end_time - start_time) / 60
    elapsed_hours = elapsed_minutes / 60

    show_progress("DADA2 denoising complete", 80)
    print("✓ DADA2 denoising completed successfully")
    print(f"  Runtime: {elapsed_minutes:.2f} minutes ({elapsed_hours:.2f} hours)")

    # Save output artifacts
    table.save(str(feature_table_path))
    rep_seqs.save(str(rep_seqs_path))
    dada2_stats.save(str(dada2_stats_path))

    show_progress("Saved QIIME artifacts", 90)

    print(f"\n✓ Saved feature table: {feature_table_path}")
    print(f"✓ Saved representative sequences: {rep_seqs_path}")
    print(f"✓ Saved DADA2 statistics: {dada2_stats_path}")

    show_progress("Feature table summary complete", 100)
    print(f"✓ Saved feature table summary: {output_dir}/feature_table_summary.qzv")
    print("\n" + "=" * 60)
    print("✓ DADA2 PIPELINE COMPLETE!")
    print("=" * 60)

Running DADA2 denoising pipeline (FULL DATASET - 39 samples)

Parameters:
  Trim left forward: 0
  Trim left reverse: 0
  Truncate length forward: 280
  Truncate length reverse: 260
  Pooling method: pseudo
  Chimera method: consensus
  Min fold parent over abundance: 8.0
  Threads: 7
  Error learning reads: 500,000 (optimized for speed)

[░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% - Starting DADA2
[███░░░░░░░░░░░░░░░░░░░░░░░░░░░]  10% - Launching denoising job
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_dada.R --input_directory /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/tmpjsx08t33/forward --input_directory_reverse /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/tmpjsx08t33/reverse --output_path /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/tmpjsx08t33/output.tsv.biom --output_trac

Loading required package: Rcpp


DADA2: 1.38.0 / Rcpp: 1.1.1 / RcppParallel: 5.1.11.2 
2) Filtering .................................................
3) Learning Error Rates
241577560 total bases in 862777 reads from 7 samples will be used for learning the error rates.
224322020 total bases in 862777 reads from 7 samples will be used for learning the error rates.
3) Denoise samples .................................................
  Pseudo-pool step .................................................
.................................................
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output


/opt/miniconda3/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan 

[████████████████████████░░░░░░]  80% - DADA2 denoising complete
✓ DADA2 denoising completed successfully
  Runtime: 2688.22 minutes (44.80 hours)
[███████████████████████████░░░]  90% - Saved QIIME artifacts

✓ Saved feature table: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/feature_table.qza
✓ Saved representative sequences: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/rep_seqs.qza
✓ Saved DADA2 statistics: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/dada2_stats.qza
[██████████████████████████████] 100% - Feature table summary complete
✓ Saved feature table summary: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/feature_table_summary.qzv

✓ DADA2 PIPELINE COMPLETE!


## Visualising qza feature table
This helps to visualise the feature table for interpretation in the Qiime viewer

In [6]:
from pathlib import Path
import subprocess
import sys

# Pick an available feature table artifact, preferring the full dataset if it exists.
table_candidates = [
    output_dir / 'feature_table.qza',
    output_dir / 'test_feature_table_3samples.qza',
]

table_qza = None
for candidate in table_candidates:
    if candidate.exists():
        table_qza = candidate
        break

if table_qza is None:
    raise FileNotFoundError(
        'No feature table artifact found. Run the DADA2 step first or place feature_table.qza in the output directory.'
    )

table_qzv = output_dir / 'table.qzv'
subprocess.run(
    [
        'qiime', 'feature-table', 'summarize',
        '--i-table', str(table_qza),
        '--o-feature-frequencies', str(output_dir / 'feature_frequencies.qza'),
        '--o-sample-frequencies', str(output_dir / 'sample_frequencies.qza'),
        '--o-summary', str(table_qzv),
    ],
    check=True,
)
print(f'✓ Saved feature table visualization: {table_qzv}')

Saved ImmutableMetadata to: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/feature_frequencies.qza
Saved ImmutableMetadata to: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/sample_frequencies.qza
Saved Visualization to: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/table.qzv
✓ Saved feature table visualization: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/table.qzv
